In [10]:
import sys
import os

project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.append(project_root)

from FEATURES.features import *
from PRODUCTION.pipeline import *
from PRODUCTION.teamInfo import teamStarPlayer, projectedStartingFive, mainStartingFive

from math import log, exp
from scipy.stats import poisson
from scipy.stats import gamma
import numpy as np
from nba_api.stats.endpoints import leaguedashteamstats

In [11]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

s25 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_25.csv').sort_values(by='GAME_DATE')
s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')

dfsData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_DFS_{today}.csv')

/var/folders/9q/5_554qsx5z70w9d_vkmvjg0h0000gn/T/ipykernel_19940/3699767750.py:6: DtypeWarning: Columns (33) have mixed types. Specify dtype option on import or set low_memory=False.
  s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')


In [12]:
def count_line_hits(player_df, line, category, game_windows=[5, 10, 15]):
    results = {}
    player_df_sorted = player_df.sort_values('GAME_DATE')
    total_games = len(player_df_sorted)

    for window in game_windows:
        # Handle players with fewer games
        if total_games < window:
            last_n_games = player_df_sorted
        else:
            last_n_games = player_df_sorted.tail(window)

        if category == 'player_points':
            hits = (last_n_games['PTS'] > line).sum()
        elif category == 'player_assists':
            hits = (last_n_games['AST'] > line).sum()
        elif category == 'player_rebounds':
            hits = (last_n_games['REB'] > line).sum()
        elif category == 'player_threes':
            hits = (last_n_games['FG3M'] > line).sum()
        elif category == 'player_blocks':
            hits = (last_n_games['BLK'] > line).sum()
        elif category == 'player_steals':
            hits = (last_n_games['STL'] > line).sum()
        elif category == 'player_field_goals':
            hits = (last_n_games['FGM'] > line).sum()
        elif category == 'player_frees_made':
            hits = (last_n_games['FTM'] > line).sum()
        elif category == 'player_points_rebounds_assists':
            hits = (last_n_games['PTS'] + last_n_games['REB'] + last_n_games['AST'] > line).sum()
        elif category == 'player_points_rebounds':
            hits = (last_n_games['PTS'] + last_n_games['REB'] > line).sum()
        elif category == 'player_points_assists':
            hits = (last_n_games['PTS'] + last_n_games['AST'] > line).sum()
        elif category == 'player_rebounds_assists':
            hits = (last_n_games['REB'] + last_n_games['AST'] > line).sum()
        elif category == 'player_turnovers':
            hits = (last_n_games['TOV'] > line).sum()
        else:
            hits = 0

        results['NAME'] = player_df_sorted['PLAYER_NAME'].iloc[0] if total_games > 0 else 'Unknown'
        results['CATEGORY'] = category
        results['LINE'] = line
        results[f'L-{window}'] = round(hits / window, 2)


    return results

# Bayesian posterior prediction.

## Points

In [13]:
dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]
res = []
PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))

league_df = leaguedashteamstats.LeagueDashTeamStats(
    league_id_nullable='00',
    per_mode_detailed='PerGame',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]

team_stats = league_df.set_index('TEAM_ID')
league_avg_def_rtg = league_df['DEF_RATING'].mean()
league_avg_off_rtg = league_df['OFF_RATING'].mean()
league_avg_pace = league_df['PACE'].mean()

def compute_bayesian_lambda(player_df, player_df_25, player_team, opp_team_id, 
                           team_stats, league_avg_off_rtg, league_avg_def_rtg, 
                           league_avg_pace, home_flag, current_date):
    """
    Compute Bayesian posterior lambda for Poisson model.
    
    Uses Gamma prior (conjugate to Poisson) with:
    - Prior mean based on historical performance and contextual factors
    - Prior strength based on sample size and confidence
    - Updates with recent observations
    """
    eplison = 1e-6
    
    # Get baseline from season data
    if len(player_df) < 5:
        baseline_mean = player_df_25['PTS'].mean() if not player_df_25.empty else 10.0
        baseline_games = len(player_df_25)
    else:
        baseline_mean = player_df['PTS'].mean()
        baseline_games = len(player_df)
    
    if baseline_mean <= 0:
        return None
    
    # Get team and opponent stats
    team_or = team_stats.at[player_team, 'OFF_RATING']
    team_pace = team_stats.at[player_team, 'PACE']
    opp_dr = team_stats.at[opp_team_id, 'DEF_RATING']
    opp_pace = team_stats.at[opp_team_id, 'PACE']
    
    # Contextual adjustments for prior mean
    # Team offensive strength
    team_or_factor = team_or / league_avg_off_rtg
    
    # Opponent defensive weakness (lower def rating = easier matchup)
    opp_dr_factor = league_avg_def_rtg / opp_dr
    
    # Pace adjustment
    expected_pace = (team_pace + opp_pace) / 2
    pace_factor = expected_pace / league_avg_pace
    
    # Home court advantage
    home_factor = 1.03 if home_flag else 0.97
    
    # Days rested
    current_date_dt = pd.to_datetime(current_date)
    player_df['GAME_DATE'] = pd.to_datetime(player_df['GAME_DATE'])
    last_game_date = player_df['GAME_DATE'].max()
    days_rested = (current_date_dt - last_game_date).days
    if days_rested == 1:
        rest_factor = 0.93
    elif days_rested == 2:
        rest_factor = 1.00
    elif 3 <= days_rested <= 5:
        rest_factor = 1.02
    else:
        rest_factor = 0.98
    
    # Head-to-head adjustment
    h2h = player_df[player_df['OPP_ABBREVIATION'] == opp_team]
    if h2h.empty and not player_df_25.empty:
        h2h = player_df_25[player_df_25['OPP_ABBREVIATION'] == opp_team]
    h2h_factor = 1.0
    if not h2h.empty:
        h2h_avg = h2h['PTS'].mean()
        h2h_factor = h2h_avg / baseline_mean if baseline_mean > 0 else 1.0
        # Cap H2H factor to prevent extreme values
        h2h_factor = max(0.85, min(1.15, h2h_factor))
    
    # Prior mean: baseline adjusted by contextual factors
    prior_mean = baseline_mean * team_or_factor * opp_dr_factor * pace_factor * home_factor * rest_factor * h2h_factor
    
    # Prior strength: how much weight we give to the prior
    # More games = stronger prior, but also consider recency
    prior_strength = min(baseline_games, 40)  # Cap at 40 games for stability
    prior_strength = max(prior_strength, 5)   # Minimum 5 games
    
    # Gamma prior parameters
    # For Gamma(alpha, beta), mean = alpha/beta
    # We want mean = prior_mean, so alpha = prior_mean * beta
    # beta controls the strength (higher beta = tighter prior)
    prior_beta = prior_strength / 10.0  # Scale: 10 games = beta of 1
    prior_alpha = prior_mean * prior_beta
    
    # Ensure prior_alpha >= 1 for valid Gamma distribution
    if prior_alpha < 1:
        prior_alpha = 1.0
        prior_beta = prior_alpha / prior_mean
    
    # Get recent observations for likelihood
    # Use last 7 games, or all available if fewer
    recent_games = min(7, len(player_df))
    recent_pts = player_df['PTS'].tail(recent_games).values
    
    # Bayesian update: Gamma is conjugate to Poisson
    # Posterior: Gamma(alpha_0 + sum(x_i), beta_0 + n)
    posterior_alpha = prior_alpha + recent_pts.sum()
    posterior_beta = prior_beta + recent_games
    
    # Posterior mean (this is our adjusted lambda)
    lambda_adjusted = posterior_alpha / posterior_beta
    
    # Also get posterior variance for uncertainty quantification
    posterior_variance = posterior_alpha / (posterior_beta ** 2)
    posterior_std = np.sqrt(posterior_variance)
    
    return lambda_adjusted, posterior_std

for _, row in PLAYERS.iterrows():
    PLAYER = row['NAME']
    target_pts = row['LINE']

    player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
    if player_df.empty:
        continue

    player_team = player_df['TEAM_ID'].iloc[-1]
    opp_team, home_flag = findOpp(PLAYER, player_df, current_date)

    opp_matches = s26[s26['TEAM_ABBREVIATION'] == opp_team]
    if opp_matches.empty:
        continue
    opp_team_id = opp_matches['TEAM_ID'].iloc[-1]

    # Get historical data for prior
    player_df_25 = s25[s25["PLAYER_NAME"] == PLAYER].copy() if 's25' in locals() else pd.DataFrame()
    
    # Compute Bayesian lambda
    result = compute_bayesian_lambda(
        player_df, player_df_25, player_team, opp_team_id,
        team_stats, league_avg_off_rtg, league_avg_def_rtg,
        league_avg_pace, home_flag, current_date
    )
    
    if result is None:
        continue
    
    lambda_adjusted, posterior_std = result

    # Determine which Poisson calculation to use based on line type
    if target_pts % 1 == 0.5:
        prob_over_poisson = poisson.sf(int(target_pts), lambda_adjusted)
        line_type = "X.5 (need {}+)".format(int(target_pts) + 1)
    elif target_pts % 1 == 0:
        prob_over_poisson = poisson.sf(target_pts, lambda_adjusted)
        line_type = "Over {} (need {}+)".format(int(target_pts), int(target_pts) + 1)
    else:
        # Handle other cases
        prob_over_poisson = poisson.sf(int(target_pts), lambda_adjusted)

    res.append({
        'NAME': PLAYER,
        'LINE': target_pts,
        'L-5': round(count_line_hits(player_df, target_pts, 'player_points', [5])['L-5'], 3),
        'L-10': round(count_line_hits(player_df, target_pts, 'player_points', [10])['L-10'], 3),
        'L-15': round(count_line_hits(player_df, target_pts, 'player_points', [15])['L-15'], 3),
        'OVER%': round(prob_over_poisson, 3),
        'UNDER%': round(1 - prob_over_poisson, 3),
    })

point_df = pd.DataFrame(res).sort_values(by='OVER%', ascending=False).reset_index(drop=True)
point_df.to_csv(f'../DATA/CSV_FILES/PROP_DATA/OVER_RATES_UNDERDOG/player_points.csv', index=False)
point_df.head(10)

ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))

## Assists

In [ ]:
dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_assists')]

res = []
PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))

league_df = leaguedashteamstats.LeagueDashTeamStats(
    league_id_nullable='00',
    per_mode_detailed='PerGame',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]

team_stats = league_df.set_index('TEAM_ID')
league_avg_def_rtg = league_df['DEF_RATING'].mean()
league_avg_pace = league_df['PACE'].mean()
league_avg_ast_ratio = league_df['AST_RATIO'].mean()
league_avg_tov = league_df['TM_TOV_PCT'].mean()

def compute_bayesian_lambda_assists(player_df, player_df_25, player_team, opp_team_id,
                                   team_stats, league_avg_def_rtg, league_avg_pace,
                                   league_avg_ast_ratio, league_avg_tov, home_flag, current_date):
    """
    Compute Bayesian posterior lambda for assists using Poisson model.
    
    Uses Gamma prior (conjugate to Poisson) with assists-specific contextual factors.
    """
    eplison = 1e-6
    
    # Get baseline from season data
    if len(player_df) < 5:
        baseline_mean = player_df_25['AST'].mean() if not player_df_25.empty else 3.0
        baseline_games = len(player_df_25)
    else:
        baseline_mean = player_df['AST'].mean()
        baseline_games = len(player_df)
    
    if baseline_mean <= 0:
        return None
    
    # Get team and opponent stats
    team_pace = team_stats.at[player_team, 'PACE']
    team_ast_ratio = team_stats.at[player_team, 'AST_RATIO']
    opp_dr = team_stats.at[opp_team_id, 'DEF_RATING']
    opp_pace = team_stats.at[opp_team_id, 'PACE']
    opp_tov = team_stats.at[opp_team_id, 'TM_TOV_PCT']
    
    # Contextual adjustments for prior mean (assists-specific)
    # Team assist culture (pass-heavy teams create more assists)
    team_ast_ratio_factor = team_ast_ratio / league_avg_ast_ratio
    
    # Opponent defensive weakness (weaker defense = easier passes)
    opp_dr_factor = league_avg_def_rtg / opp_dr
    
    # Opponent turnover pressure (teams that force turnovers limit assists)
    opp_tov_factor = league_avg_tov / opp_tov
    
    # Pace adjustment (more possessions = more assist opportunities)
    expected_pace = (team_pace + opp_pace) / 2
    pace_factor = expected_pace / league_avg_pace
    
    # Home court advantage (smaller effect for assists)
    home_factor = 1.02 if home_flag else 0.98
    
    # Days rested
    current_date_dt = pd.to_datetime(current_date)
    player_df['GAME_DATE'] = pd.to_datetime(player_df['GAME_DATE'])
    last_game_date = player_df['GAME_DATE'].max()
    days_rested = (current_date_dt - last_game_date).days
    if days_rested == 1:
        rest_factor = 0.93
    elif days_rested == 2:
        rest_factor = 1.00
    elif 3 <= days_rested <= 5:
        rest_factor = 1.02
    else:
        rest_factor = 0.98
    
    # Head-to-head adjustment
    h2h = player_df[player_df['OPP_ABBREVIATION'] == opp_team]
    if h2h.empty and not player_df_25.empty:
        h2h = player_df_25[player_df_25['OPP_ABBREVIATION'] == opp_team]
    h2h_factor = 1.0
    if not h2h.empty:
        h2h_avg = h2h['AST'].mean()
        h2h_factor = h2h_avg / baseline_mean if baseline_mean > 0 else 1.0
        h2h_factor = max(0.85, min(1.15, h2h_factor))
    
    # Prior mean: baseline adjusted by contextual factors
    prior_mean = (baseline_mean * team_ast_ratio_factor * opp_dr_factor * 
                  opp_tov_factor * pace_factor * home_factor * rest_factor * h2h_factor)
    
    # Prior strength: how much weight we give to the prior
    prior_strength = min(baseline_games, 40)
    prior_strength = max(prior_strength, 5)
    
    # Gamma prior parameters
    prior_beta = prior_strength / 10.0
    prior_alpha = prior_mean * prior_beta
    
    # Ensure valid Gamma distribution
    if prior_alpha < 1:
        prior_alpha = 1.0
        prior_beta = prior_alpha / prior_mean
    
    # Get recent observations for likelihood (last 7 games)
    recent_games = min(7, len(player_df))
    recent_ast = player_df['AST'].tail(recent_games).values
    
    # Bayesian update: Gamma is conjugate to Poisson
    # Posterior: Gamma(alpha_0 + sum(x_i), beta_0 + n)
    posterior_alpha = prior_alpha + recent_ast.sum()
    posterior_beta = prior_beta + recent_games
    
    # Posterior mean (adjusted lambda)
    lambda_adjusted = posterior_alpha / posterior_beta
    
    # Posterior variance for uncertainty
    posterior_variance = posterior_alpha / (posterior_beta ** 2)
    posterior_std = np.sqrt(posterior_variance)
    
    return lambda_adjusted, posterior_std

for _, row in PLAYERS.iterrows():
    PLAYER = row['NAME']
    target_ast = row['LINE']

    player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
    if player_df.empty:
        continue

    player_team = player_df['TEAM_ID'].iloc[-1]
    opp_team, home_flag = findOpp(PLAYER, player_df, current_date)

    opp_matches = s26[s26['TEAM_ABBREVIATION'] == opp_team]
    if opp_matches.empty:
        continue
    opp_team_id = opp_matches['TEAM_ID'].iloc[-1]

    # Get historical data for prior
    player_df_25 = s25[s25["PLAYER_NAME"] == PLAYER].copy() if 's25' in locals() else pd.DataFrame()
    
    # Compute Bayesian lambda
    result = compute_bayesian_lambda_assists(
        player_df, player_df_25, player_team, opp_team_id,
        team_stats, league_avg_def_rtg, league_avg_pace,
        league_avg_ast_ratio, league_avg_tov, home_flag, current_date
    )
    
    if result is None:
        continue
    
    lambda_adjusted, posterior_std = result

    # Determine which Poisson calculation to use based on line type
    if target_ast % 1 == 0.5:
        prob_over_poisson = poisson.sf(int(target_ast), lambda_adjusted)
        line_type = "X.5 (need {}+)".format(int(target_ast) + 1)
    elif target_ast % 1 == 0:
        prob_over_poisson = poisson.sf(target_ast, lambda_adjusted)
        line_type = "Over {} (need {}+)".format(int(target_ast), int(target_ast) + 1)
    else:
        prob_over_poisson = poisson.sf(int(target_ast), lambda_adjusted)

    res.append({
        'NAME': PLAYER,
        'LINE': target_ast,
        'L-5': round(count_line_hits(player_df, target_ast, 'player_assists', [5])['L-5'], 3),
        'L-10': round(count_line_hits(player_df, target_ast, 'player_assists', [10])['L-10'], 3),
        'L-15': round(count_line_hits(player_df, target_ast, 'player_assists', [15])['L-15'], 3),
        'OVER%': round(prob_over_poisson, 3),
        'UNDER%': round(1 - prob_over_poisson, 3),
    })
    
assist_df = pd.DataFrame(res).sort_values(by='OVER%', ascending=False).reset_index(drop=True)
assist_df.to_csv(f'../DATA/CSV_FILES/PROP_DATA/OVER_RATES_UNDERDOG/player_assists.csv', index=False)
assist_df.head(10)

,NAME,LINE,L-5,L-10,L-15,OVER%,UNDER%
0,Paul George,2.5,0.2,0.1,0.07,0.832,0.168
1,Jrue Holiday,6.5,0.6,0.6,0.53,0.739,0.261
2,Alperen Sengun,6.5,0.8,0.8,0.67,0.614,0.386
3,Kevin Durant,3.5,0.6,0.5,0.33,0.612,0.388
4,Julius Randle,5.5,0.6,0.5,0.60,0.607,0.393
5,Cole Anthony,4.5,0.6,0.6,0.60,0.594,0.406
6,Al Horford,1.5,0.4,0.5,0.33,0.593,0.407
7,Terance Mann,3.5,0.6,0.7,0.47,0.562,0.438
8,Luke Kennard,1.5,0.6,0.7,0.67,0.547,0.453
9,Amen Thompson,4.5,0.6,0.6,0.53,0.547,0.453


# REBOUNDS

In [ ]:
dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_rebounds')]

res = []
PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))

league_df = leaguedashteamstats.LeagueDashTeamStats(
    league_id_nullable='00',
    per_mode_detailed='PerGame',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]

team_stats = league_df.set_index('TEAM_ID')
league_avg_pace = league_df['PACE'].mean()
league_avg_oreb = league_df['OREB_PCT'].mean()
league_avg_dreb = league_df['DREB_PCT'].mean()
league_avg_reb = league_df['REB_PCT'].mean()

def compute_bayesian_lambda_rebounds(player_df, player_df_25, player_team, opp_team_id,
                                     team_stats, league_avg_pace, league_avg_reb,
                                     league_avg_oreb, league_avg_dreb, home_flag, current_date):
    """
    Compute Bayesian posterior lambda for rebounds using Poisson model.
    
    Uses Gamma prior (conjugate to Poisson) with rebounds-specific contextual factors.
    """
    eplison = 1e-6
    
    # Get baseline from season data
    if len(player_df) < 5:
        baseline_mean = player_df_25['REB'].mean() if not player_df_25.empty else 5.0
        baseline_games = len(player_df_25)
    else:
        baseline_mean = player_df['REB'].mean()
        baseline_games = len(player_df)
    
    if baseline_mean <= 0:
        return None
    
    # Get team and opponent stats
    team_pace = team_stats.at[player_team, 'PACE']
    team_reb = team_stats.at[player_team, 'REB_PCT']
    team_oreb = team_stats.at[player_team, 'OREB_PCT']
    team_dreb = team_stats.at[player_team, 'DREB_PCT']
    opp_pace = team_stats.at[opp_team_id, 'PACE']
    opp_reb = team_stats.at[opp_team_id, 'REB_PCT']
    opp_oreb = team_stats.at[opp_team_id, 'OREB_PCT']
    opp_dreb = team_stats.at[opp_team_id, 'DREB_PCT']
    
    # Contextual adjustments for prior mean (rebounds-specific)
    # Team rebounding culture (rebounding-focused teams)
    team_reb_factor = team_reb / league_avg_reb
    
    # Opponent gives up offensive rebounds (weak DREB% = more offensive boards available)
    opp_dreb_factor = league_avg_dreb / opp_dreb
    
    # Opponent gives up defensive rebounds (weak OREB% = more defensive boards available)
    opp_oreb_factor = league_avg_oreb / opp_oreb
    
    # Pace adjustment (more possessions = more missed shots = more rebounds)
    expected_pace = (team_pace + opp_pace) / 2
    pace_factor = expected_pace / league_avg_pace
    
    # Home court advantage (minimal effect for rebounds)
    home_factor = 1.01 if home_flag else 0.99
    
    # Days rested
    current_date_dt = pd.to_datetime(current_date)
    player_df['GAME_DATE'] = pd.to_datetime(player_df['GAME_DATE'])
    last_game_date = player_df['GAME_DATE'].max()
    days_rested = (current_date_dt - last_game_date).days
    if days_rested == 1:
        rest_factor = 0.93
    elif days_rested == 2:
        rest_factor = 1.00
    elif 3 <= days_rested <= 5:
        rest_factor = 1.02
    else:
        rest_factor = 0.98
    
    # Head-to-head adjustment
    h2h = player_df[player_df['OPP_ABBREVIATION'] == opp_team]
    if h2h.empty and not player_df_25.empty:
        h2h = player_df_25[player_df_25['OPP_ABBREVIATION'] == opp_team]
    h2h_factor = 1.0
    if not h2h.empty:
        h2h_avg = h2h['REB'].mean()
        h2h_factor = h2h_avg / baseline_mean if baseline_mean > 0 else 1.0
        h2h_factor = max(0.85, min(1.15, h2h_factor))
    
    # Prior mean: baseline adjusted by contextual factors
    prior_mean = (baseline_mean * team_reb_factor * opp_dreb_factor * 
                  opp_oreb_factor * pace_factor * home_factor * rest_factor * h2h_factor)
    
    # Prior strength: how much weight we give to the prior
    prior_strength = min(baseline_games, 40)
    prior_strength = max(prior_strength, 5)
    
    # Gamma prior parameters
    prior_beta = prior_strength / 10.0
    prior_alpha = prior_mean * prior_beta
    
    # Ensure valid Gamma distribution
    if prior_alpha < 1:
        prior_alpha = 1.0
        prior_beta = prior_alpha / prior_mean
    
    # Get recent observations for likelihood (last 7 games)
    recent_games = min(7, len(player_df))
    recent_reb = player_df['REB'].tail(recent_games).values
    
    # Bayesian update: Gamma is conjugate to Poisson
    # Posterior: Gamma(alpha_0 + sum(x_i), beta_0 + n)
    posterior_alpha = prior_alpha + recent_reb.sum()
    posterior_beta = prior_beta + recent_games
    
    # Posterior mean (adjusted lambda)
    lambda_adjusted = posterior_alpha / posterior_beta
    
    # Posterior variance for uncertainty
    posterior_variance = posterior_alpha / (posterior_beta ** 2)
    posterior_std = np.sqrt(posterior_variance)
    
    return lambda_adjusted, posterior_std

for _, row in PLAYERS.iterrows():
    PLAYER = row['NAME']
    target_reb = row['LINE']

    player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
    if player_df.empty:
        continue

    player_team = player_df['TEAM_ID'].iloc[-1]
    opp_team, home_flag = findOpp(PLAYER, player_df, current_date)

    opp_matches = s26[s26['TEAM_ABBREVIATION'] == opp_team]
    if opp_matches.empty:
        continue
    opp_team_id = opp_matches['TEAM_ID'].iloc[-1]

    # Get historical data for prior
    player_df_25 = s25[s25["PLAYER_NAME"] == PLAYER].copy() if 's25' in locals() else pd.DataFrame()
    
    # Compute Bayesian lambda
    result = compute_bayesian_lambda_rebounds(
        player_df, player_df_25, player_team, opp_team_id,
        team_stats, league_avg_pace, league_avg_reb,
        league_avg_oreb, league_avg_dreb, home_flag, current_date
    )
    
    if result is None:
        continue
    
    lambda_adjusted, posterior_std = result

    # Determine which Poisson calculation to use based on line type
    if target_reb % 1 == 0.5:
        prob_over_poisson = poisson.sf(int(target_reb), lambda_adjusted)
        line_type = "X.5 (need {}+)".format(int(target_reb) + 1)
    elif target_reb % 1 == 0:
        prob_over_poisson = poisson.sf(target_reb, lambda_adjusted)
        line_type = "Over {} (need {}+)".format(int(target_reb), int(target_reb) + 1)
    else:
        prob_over_poisson = poisson.sf(int(target_reb), lambda_adjusted)

    res.append({
        'NAME': PLAYER,
        'LINE': target_reb,
        'L-5': round(count_line_hits(player_df, target_reb, 'player_rebounds', [5])['L-5'], 3),
        'L-10': round(count_line_hits(player_df, target_reb, 'player_rebounds', [10])['L-10'], 3),
        'L-15': round(count_line_hits(player_df, target_reb, 'player_rebounds', [15])['L-15'], 3),
        'OVER%': round(prob_over_poisson, 3),
        'UNDER%': round(1 - prob_over_poisson, 3),
    })
    
rebound_df = pd.DataFrame(res).sort_values(by='OVER%', ascending=False).reset_index(drop=True)
rebound_df.to_csv(f'../DATA/CSV_FILES/PROP_DATA/OVER_RATES_UNDERDOG/player_rebounds.csv', index=False)
rebound_df.head(10)

,NAME,LINE,L-5,L-10,L-15,OVER%,UNDER%
0,Matas Buzelis,4.5,0.8,0.7,0.67,0.766,0.234
1,Alperen Sengun,9.5,1.0,0.8,0.60,0.691,0.309
2,Cedric Coward,6.5,0.6,0.6,0.40,0.685,0.315
3,Trey Murphy III,5.5,0.4,0.5,0.53,0.669,0.331
4,Josh Giddey,9.5,0.4,0.6,0.40,0.655,0.345
5,Brandon Williams,2.5,0.6,0.4,0.33,0.644,0.356
6,Cooper Flagg,5.5,0.6,0.7,0.60,0.594,0.406
7,Isaac Okoro,2.5,0.4,0.4,0.47,0.594,0.406
8,Jrue Holiday,4.5,0.6,0.6,0.53,0.591,0.409
9,Brandin Podziemski,4.5,0.6,0.5,0.53,0.546,0.454


## Blocks

In [ ]:
dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_blocks')]

res = []
PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))

league_df = leaguedashteamstats.LeagueDashTeamStats(
    league_id_nullable='00',
    per_mode_detailed='PerGame',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]

team_stats = league_df.set_index('TEAM_ID')
league_avg_pace = league_df['PACE'].mean()

def compute_bayesian_lambda_blocks(player_df, player_df_25, player_team, opp_team_id,
                                   team_stats, league_avg_pace, home_flag, current_date):
    """
    Compute Bayesian posterior lambda for blocks using Poisson model.
    
    Uses Gamma prior (conjugate to Poisson) with blocks-specific contextual factors.
    Blocks are rare events, so we use a more conservative prior.
    """
    eplison = 1e-6
    
    # Get baseline from season data
    if len(player_df) < 5:
        baseline_mean = player_df_25['BLK'].mean() if not player_df_25.empty else 0.5
        baseline_games = len(player_df_25)
    else:
        baseline_mean = player_df['BLK'].mean()
        baseline_games = len(player_df)
    
    if baseline_mean <= 0:
        baseline_mean = 0.1  # Minimum for blocks (very rare events)
    
    # Get team and opponent stats
    team_pace = team_stats.at[player_team, 'PACE']
    opp_pace = team_stats.at[opp_team_id, 'PACE']
    
    # Contextual adjustments for prior mean (blocks-specific)
    # Pace adjustment (more possessions = more shot attempts = more block opportunities)
    expected_pace = (team_pace + opp_pace) / 2
    pace_factor = expected_pace / league_avg_pace
    
    # Home court advantage (minimal effect for blocks)
    home_factor = 1.00  # No home/away effect for blocks
    
    # Days rested
    current_date_dt = pd.to_datetime(current_date)
    player_df['GAME_DATE'] = pd.to_datetime(player_df['GAME_DATE'])
    last_game_date = player_df['GAME_DATE'].max()
    days_rested = (current_date_dt - last_game_date).days
    if days_rested == 1:
        rest_factor = 0.93
    elif days_rested == 2:
        rest_factor = 1.00
    elif 3 <= days_rested <= 5:
        rest_factor = 1.02
    else:
        rest_factor = 0.98
    
    # Head-to-head adjustment
    h2h = player_df[player_df['OPP_ABBREVIATION'] == opp_team]
    if h2h.empty and not player_df_25.empty:
        h2h = player_df_25[player_df_25['OPP_ABBREVIATION'] == opp_team]
    h2h_factor = 1.0
    if not h2h.empty:
        h2h_avg = h2h['BLK'].mean()
        h2h_factor = h2h_avg / baseline_mean if baseline_mean > 0 else 1.0
        h2h_factor = max(0.80, min(1.20, h2h_factor))  # Wider range for rare events
    
    # Prior mean: baseline adjusted by contextual factors
    prior_mean = baseline_mean * pace_factor * home_factor * rest_factor * h2h_factor
    
    # Prior strength: for rare events like blocks, use more conservative prior
    # Blocks are more volatile, so we give less weight to prior
    prior_strength = min(baseline_games, 30)  # Cap lower for blocks
    prior_strength = max(prior_strength, 3)   # Minimum 3 games
    
    # Gamma prior parameters
    prior_beta = prior_strength / 8.0  # Lower beta = weaker prior (more weight to data)
    prior_alpha = prior_mean * prior_beta
    
    # Ensure valid Gamma distribution
    if prior_alpha < 0.1:  # Lower minimum for rare events
        prior_alpha = 0.1
        prior_beta = prior_alpha / prior_mean
    
    # Get recent observations for likelihood (last 7 games, or more for rare events)
    recent_games = min(10, len(player_df))  # Use more games for blocks (rare events)
    recent_blk = player_df['BLK'].tail(recent_games).values
    
    # Bayesian update: Gamma is conjugate to Poisson
    # Posterior: Gamma(alpha_0 + sum(x_i), beta_0 + n)
    posterior_alpha = prior_alpha + recent_blk.sum()
    posterior_beta = prior_beta + recent_games
    
    # Posterior mean (adjusted lambda)
    lambda_adjusted = posterior_alpha / posterior_beta
    
    # Ensure lambda is positive
    if lambda_adjusted <= 0:
        lambda_adjusted = 0.1
    
    # Posterior variance for uncertainty
    posterior_variance = posterior_alpha / (posterior_beta ** 2)
    posterior_std = np.sqrt(posterior_variance)
    
    return lambda_adjusted, posterior_std

for _, row in PLAYERS.iterrows():
    PLAYER = row['NAME']
    target_blk = row['LINE']

    player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
    if player_df.empty:
        continue

    player_team = player_df['TEAM_ID'].iloc[-1]
    opp_team, home_flag = findOpp(PLAYER, player_df, current_date)

    opp_matches = s26[s26['TEAM_ABBREVIATION'] == opp_team]
    if opp_matches.empty:
        continue
    opp_team_id = opp_matches['TEAM_ID'].iloc[-1]

    # Get historical data for prior
    player_df_25 = s25[s25["PLAYER_NAME"] == PLAYER].copy() if 's25' in locals() else pd.DataFrame()
    
    # Compute Bayesian lambda
    result = compute_bayesian_lambda_blocks(
        player_df, player_df_25, player_team, opp_team_id,
        team_stats, league_avg_pace, home_flag, current_date
    )
    
    if result is None:
        continue
    
    lambda_adjusted, posterior_std = result

    # Determine which Poisson calculation to use based on line type
    if target_blk % 1 == 0.5:
        prob_over_poisson = poisson.sf(int(target_blk), lambda_adjusted)
        line_type = "X.5 (need {}+)".format(int(target_blk) + 1)
    elif target_blk % 1 == 0:
        prob_over_poisson = poisson.sf(target_blk, lambda_adjusted)
        line_type = "Over {} (need {}+)".format(int(target_blk), int(target_blk) + 1)
    else:
        prob_over_poisson = poisson.sf(int(target_blk), lambda_adjusted)

    res.append({
        'NAME': PLAYER,
        'LINE': target_blk,
        'L-5': round(count_line_hits(player_df, target_blk, 'player_blocks', [5])['L-5'], 3),
        'L-10': round(count_line_hits(player_df, target_blk, 'player_blocks', [10])['L-10'], 3),
        'L-15': round(count_line_hits(player_df, target_blk, 'player_blocks', [15])['L-15'], 3),
        'OVER%': round(prob_over_poisson, 3),
        'UNDER%': round(1 - prob_over_poisson, 3),
        'IMPLIED_ODDS': round(1 / prob_over_poisson, 3) if prob_over_poisson > 0 else None,
    })
    
blocks_df = pd.DataFrame(res)
blocks_df.to_csv(f'../DATA/CSV_FILES/PROP_DATA/OVER_RATES_UNDERDOG/player_blocks.csv', index=False)
blocks_df.head(10)

,NAME,LINE,L-5,L-10,L-15,OVER%,UNDER%,IMPLIED_ODDS
0,Scottie Barnes,1.5,0.4,0.6,0.47,0.610,0.390,1.640
1,Alex Sarr,1.5,0.6,0.8,0.67,0.652,0.348,1.533
2,Donovan Clingan,1.5,0.8,0.6,0.53,0.485,0.515,2.063


# STEALS

In [ ]:
dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_steals')]

res = []
PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))

league_df = leaguedashteamstats.LeagueDashTeamStats(
    league_id_nullable='00',
    per_mode_detailed='PerGame',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]

team_stats = league_df.set_index('TEAM_ID')
league_avg_pace = league_df['PACE'].mean()
league_avg_tov = league_df['TM_TOV_PCT'].mean()

def compute_bayesian_lambda_steals(player_df, player_df_25, player_team, opp_team_id,
                                   team_stats, league_avg_pace, league_avg_tov,
                                   home_flag, current_date):
    """
    Compute Bayesian posterior lambda for steals using Poisson model.
    
    Uses Gamma prior (conjugate to Poisson) with steals-specific contextual factors.
    Steals are rare events, so we use a more conservative prior similar to blocks.
    """
    eplison = 1e-6
    
    # Get baseline from season data
    if len(player_df) < 5:
        baseline_mean = player_df_25['STL'].mean() if not player_df_25.empty else 0.8
        baseline_games = len(player_df_25)
    else:
        baseline_mean = player_df['STL'].mean()
        baseline_games = len(player_df)
    
    if baseline_mean <= 0:
        baseline_mean = 0.1  # Minimum for steals (rare events)
    
    # Get team and opponent stats
    team_pace = team_stats.at[player_team, 'PACE']
    opp_pace = team_stats.at[opp_team_id, 'PACE']
    opp_tov = team_stats.at[opp_team_id, 'TM_TOV_PCT']
    
    # Contextual adjustments for prior mean (steals-specific)
    # Opponent turnover rate (higher TOV% = more steal opportunities)
    opp_tov_factor = opp_tov / league_avg_tov
    
    # Pace adjustment (more possessions = more steal opportunities)
    expected_pace = (team_pace + opp_pace) / 2
    pace_factor = expected_pace / league_avg_pace
    
    # Home court advantage (minimal effect for steals)
    home_factor = 1.00  # No home/away effect for steals
    
    # Days rested
    current_date_dt = pd.to_datetime(current_date)
    player_df['GAME_DATE'] = pd.to_datetime(player_df['GAME_DATE'])
    last_game_date = player_df['GAME_DATE'].max()
    days_rested = (current_date_dt - last_game_date).days
    if days_rested == 1:
        rest_factor = 0.93
    elif days_rested == 2:
        rest_factor = 1.00
    elif 3 <= days_rested <= 5:
        rest_factor = 1.02
    else:
        rest_factor = 0.98
    
    # Head-to-head adjustment
    h2h = player_df[player_df['OPP_ABBREVIATION'] == opp_team]
    if h2h.empty and not player_df_25.empty:
        h2h = player_df_25[player_df_25['OPP_ABBREVIATION'] == opp_team]
    h2h_factor = 1.0
    if not h2h.empty:
        h2h_avg = h2h['STL'].mean()
        h2h_factor = h2h_avg / baseline_mean if baseline_mean > 0 else 1.0
        h2h_factor = max(0.80, min(1.20, h2h_factor))  # Wider range for rare events
    
    # Prior mean: baseline adjusted by contextual factors
    prior_mean = baseline_mean * opp_tov_factor * pace_factor * home_factor * rest_factor * h2h_factor
    
    # Prior strength: for rare events like steals, use more conservative prior
    prior_strength = min(baseline_games, 30)  # Cap lower for rare events
    prior_strength = max(prior_strength, 3)   # Minimum 3 games
    
    # Gamma prior parameters
    prior_beta = prior_strength / 8.0  # Lower beta = weaker prior (more weight to data)
    prior_alpha = prior_mean * prior_beta
    
    # Ensure valid Gamma distribution
    if prior_alpha < 0.1:  # Lower minimum for rare events
        prior_alpha = 0.1
        prior_beta = prior_alpha / prior_mean
    
    # Get recent observations for likelihood (last 7-10 games for rare events)
    recent_games = min(10, len(player_df))  # Use more games for steals (rare events)
    recent_stl = player_df['STL'].tail(recent_games).values
    
    # Bayesian update: Gamma is conjugate to Poisson
    # Posterior: Gamma(alpha_0 + sum(x_i), beta_0 + n)
    posterior_alpha = prior_alpha + recent_stl.sum()
    posterior_beta = prior_beta + recent_games
    
    # Posterior mean (adjusted lambda)
    lambda_adjusted = posterior_alpha / posterior_beta
    
    # Ensure lambda is positive
    if lambda_adjusted <= 0:
        lambda_adjusted = 0.1
    
    # Posterior variance for uncertainty
    posterior_variance = posterior_alpha / (posterior_beta ** 2)
    posterior_std = np.sqrt(posterior_variance)
    
    return lambda_adjusted, posterior_std

for _, row in PLAYERS.iterrows():
    PLAYER = row['NAME']
    target_stl = row['LINE']

    player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
    if player_df.empty:
        continue

    player_team = player_df['TEAM_ID'].iloc[-1]
    opp_team, home_flag = findOpp(PLAYER, player_df, current_date)

    opp_matches = s26[s26['TEAM_ABBREVIATION'] == opp_team]
    if opp_matches.empty:
        continue
    opp_team_id = opp_matches['TEAM_ID'].iloc[-1]

    # Get historical data for prior
    player_df_25 = s25[s25["PLAYER_NAME"] == PLAYER].copy() if 's25' in locals() else pd.DataFrame()
    
    # Compute Bayesian lambda
    result = compute_bayesian_lambda_steals(
        player_df, player_df_25, player_team, opp_team_id,
        team_stats, league_avg_pace, league_avg_tov, home_flag, current_date
    )
    
    if result is None:
        continue
    
    lambda_adjusted, posterior_std = result

    # Determine which Poisson calculation to use based on line type
    if target_stl % 1 == 0.5:
        prob_over_poisson = poisson.sf(int(target_stl), lambda_adjusted)
        line_type = "X.5 (need {}+)".format(int(target_stl) + 1)
    elif target_stl % 1 == 0:
        prob_over_poisson = poisson.sf(target_stl, lambda_adjusted)
        line_type = "Over {} (need {}+)".format(int(target_stl), int(target_stl) + 1)
    else:
        prob_over_poisson = poisson.sf(int(target_stl), lambda_adjusted)

    res.append({
        'NAME': PLAYER,
        'LINE': target_stl,
        'L-5': round(count_line_hits(player_df, target_stl, 'player_steals', [5])['L-5'], 3),
        'L-10': round(count_line_hits(player_df, target_stl, 'player_steals', [10])['L-10'], 3),
        'L-15': round(count_line_hits(player_df, target_stl, 'player_steals', [15])['L-15'], 3),
        'OVER%': round(prob_over_poisson, 3),
        'UNDER%': round(1 - prob_over_poisson, 3),
    })
    
steals_df = pd.DataFrame(res)
steals_df.to_csv(f'../DATA/CSV_FILES/PROP_DATA/OVER_RATES_UNDERDOG/player_steals.csv', index=False)
steals_df.head(10)

,NAME,LINE,L-5,L-10,L-15,OVER%,UNDER%
0,Kris Dunn,1.5,0.4,0.6,0.47,0.500,0.500
1,Dyson Daniels,2.5,0.6,0.5,0.47,0.413,0.587
2,Immanuel Quickley,1.5,0.6,0.5,0.53,0.364,0.636
3,Amen Thompson,1.5,0.6,0.5,0.33,0.526,0.474


In [ ]:
## COMBO PROPS - All Categories

combo_categories = [
    'player_points_rebounds_assists',
    'player_points_rebounds',
    'player_points_assists',
    'player_rebounds_assists',
    'player_turnovers',
    'player_blocks_steals'
]

for category in combo_categories:
    print(f"\nProcessing {category}...")
    
    dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == category)]
    
    if dfs_data.empty:
        print(f"No data found for {category}")
        continue
    
    res = []
    PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))
    
    for _, row in PLAYERS.iterrows():
        PLAYER = row['NAME']
        target_line = row['LINE']

        player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
        if player_df.empty:
            continue

        res.append({
            'NAME': PLAYER,
            'LINE': target_line,
            'L-5': count_line_hits(player_df, target_line, category, [5])['L-5'],
            'L-10': count_line_hits(player_df, target_line, category, [10])['L-10'],
            'L-15': count_line_hits(player_df, target_line, category, [15])['L-15'],
        })
    
    if res:
        combo_df = pd.DataFrame(res).sort_values(by='L-5', ascending=False).reset_index(drop=True)
        combo_df.to_csv(f'../DATA/CSV_FILES/PROP_DATA/OVER_RATES_UNDERDOG/{category}.csv', index=False)
        print(f"Saved {len(combo_df)} players for {category}")
        display(combo_df.head(10))
    else:
        print(f"No results for {category}")


Processing player_points_rebounds_assists...
Saved 121 players for player_points_rebounds_assists


,NAME,LINE,L-5,L-10,L-15
0,Kel'el Ware,21.5,1.0,0.7,0.60
1,Jeremiah Fears,19.5,1.0,0.8,0.67
2,Zaccharie Risacher,15.5,1.0,0.7,0.53
3,Julius Randle,35.5,0.8,0.8,0.73
4,Dillon Brooks,20.5,0.8,0.7,0.47
5,Davion Mitchell,20.5,0.8,0.6,0.53
6,Norman Powell,32.5,0.8,0.5,0.40
7,Onyeka Okongwu,23.5,0.8,0.7,0.80
8,Naz Reid,22.5,0.8,0.5,0.47
9,Klay Thompson,15.5,0.8,0.5,0.40



Processing player_points_rebounds...
Saved 56 players for player_points_rebounds


,NAME,LINE,L-5,L-10,L-15
0,Kel'el Ware,21.5,1.0,0.6,0.53
1,James Harden,33.5,0.8,0.5,0.40
2,Derrick White,21.5,0.8,0.4,0.40
3,Deni Avdija,31.5,0.8,0.8,0.53
4,Aaron Gordon,21.5,0.8,0.6,0.47
5,Alperen Sengun,31.5,0.8,0.7,0.53
6,Naz Reid,20.5,0.8,0.6,0.53
7,Julius Randle,29.5,0.8,0.7,0.73
8,Immanuel Quickley,22.5,0.8,0.6,0.47
9,Shaedon Sharpe,29.5,0.8,0.6,0.40



Processing player_points_assists...
Saved 41 players for player_points_assists


...

,NAME,LINE,L-5,L-10,L-15
0,Kel'el Ware,10.5,1.0,0.7,0.47
1,Alperen Sengun,16.5,1.0,1.0,0.73
2,Franz Wagner,10.5,0.8,0.7,0.47
3,Cole Anthony,7.5,0.8,0.7,0.60
4,Scottie Barnes,13.5,0.8,0.7,0.60
5,Russell Westbrook,14.5,0.8,0.8,0.53
6,Julius Randle,12.5,0.8,0.6,0.53
7,Zion Williamson,10.5,0.6,0.4,0.27
8,Cooper Flagg,9.5,0.6,0.6,0.60
9,P.J. Washington,8.5,0.6,0.7,0.73



Processing player_turnovers...
Saved 7 players for player_turnovers


,NAME,LINE,L-5,L-10,L-15
0,Tyrese Maxey,2.5,0.8,0.7,0.53
1,Brandon Ingram,2.5,0.6,0.4,0.53
2,Josh Giddey,3.5,0.6,0.6,0.47
3,Alperen Sengun,2.5,0.6,0.7,0.67
4,Stephen Curry,2.5,0.6,0.5,0.40
5,Devin Booker,3.5,0.4,0.5,0.47
6,Deni Avdija,3.5,0.4,0.4,0.40



Processing player_blocks_steals...
Saved 3 players for player_blocks_steals


,NAME,LINE,L-5,L-10,L-15
0,Myles Turner,2.5,0.0,0.0,0.0
1,Derik Queen,2.5,0.0,0.0,0.0
2,Rudy Gobert,2.5,0.0,0.0,0.0
